# How To Use PDELie v0.6

This notebook is a quick tour of the shipped `v0.6` surface.

It focuses on:

- the stable Heat verification path
- the `v0.6` runtime discovery utilities
- portability helpers for `GeneratorFamily`
- where to look next in the tutorial notebooks


In [6]:
import importlib.metadata as md

import numpy as np

from pdelie.data import generate_heat_1d_field_batch
from pdelie.derivatives import compute_spectral_fd_derivatives
from pdelie.discovery import to_pysindy_trajectories
from pdelie.portability import coerce_generator_family, export_generator_family_manifest
from pdelie.residuals import HeatResidualEvaluator
from pdelie.symmetry import fit_translation_generator, render_generator_family
from pdelie.verification import verify_translation_generator

print("pdelie version:", md.version("pdelie"))


pdelie version: 0.6.0


## 1. Stable Heat vertical slice

This is the same core stable path that the packaged example uses: generate synthetic Heat data, compute derivatives, evaluate the residual, fit the translation generator, and verify on held-out data.


In [7]:
training = generate_heat_1d_field_batch(batch_size=4, num_times=33, num_points=64, seed=100)
heldout = generate_heat_1d_field_batch(batch_size=3, num_times=33, num_points=64, seed=101)

derivatives = compute_spectral_fd_derivatives(training)
residual_evaluator = HeatResidualEvaluator()
generator = fit_translation_generator(training, residual_evaluator)
report = verify_translation_generator(heldout, generator, residual_evaluator)

print("derivative backend:", derivatives.backend)
print("verification classification:", report.classification)
print("generator rendering:")
for line in render_generator_family(generator):
    print("  ", line)


derivative backend: spectral_fd
verification classification: exact
generator rendering:
   X_1 = ∂x + 7.94481247536e-06t∂x + 5.65221374188e-09x∂x + 4.77594080077e-06u∂x


## 2. Portability helpers

The `v0.5` portability helpers remain part of the shipped `v0.6` surface.


In [8]:
manifest = export_generator_family_manifest(generator)
round_tripped = coerce_generator_family(manifest)

print("manifest schema version:", manifest["manifest_schema_version"])
print("generator schema version:", manifest["generator_family"]["schema_version"])
print("round-trip preserved canonical meaning:", generator.to_dict() == round_tripped.to_dict())


manifest schema version: 0.1
generator schema version: 0.2
round-trip preserved canonical meaning: True


## 3. Discovery bridge

The discovery bridge is still narrow and backend-specific. It turns a canonical `FieldBatch` into flattened trajectories for the current PySINDy path.


In [9]:
trajectories, time_values, feature_names = to_pysindy_trajectories(training)

print("num trajectories:", len(trajectories))
print("trajectory shape:", trajectories[0].shape)
print("time values shape:", time_values.shape)
print("first five feature names:")
for name in feature_names[:5]:
    print("  ", name)


num trajectories: 4
trajectory shape: (33, 64)
time values shape: (33,)
first five feature names:
   u__x_index_0
   u__x_index_1
   u__x_index_2
   u__x_index_3
   u__x_index_4


## 4. Optional downstream discovery fit

If `pysindy` and `scikit-learn` are installed, you can also run the thin runtime-only discovery adapter.


In [10]:
try:
    from pdelie.discovery import fit_pysindy_discovery

    fit_result = fit_pysindy_discovery(trajectories, time_values, feature_names)
    print("fit status:", fit_result["status"])
    print("backend:", fit_result["backend"])
    print("coefficient matrix shape:", fit_result["coefficients"].shape if fit_result["coefficients"] is not None else None)
    primary_feature = feature_names[0]
    print("primary feature:", primary_feature)
    print("primary equation terms:", fit_result["equation_terms"].get(primary_feature))
except ImportError as exc:
    print("Downstream discovery extras are not installed:")
    print(exc)


fit status: success
backend: pysindy
coefficient matrix shape: (64, 2145)
primary feature: u__x_index_0
primary equation terms: {}


## 5. Where to go next

The rest of the notebooks in `notebooks/` are focused explorations:

- raw vs translation-canonical discovery
- robustness sweeps
- portability round-trips
- discovered vs known generators
- closure and algebra diagnostics
